# Module 1: Context Engineering Walkthrough

This notebook walks through the core ideas of Module 1 interactively.
Run each cell and observe how context quality affects answer quality.


In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()

client = Anthropic()
MODEL  = 'claude-haiku-4-5-20251001'
print('Client ready.')

## 1. Poor Context vs Rich Context

Same question, two very different system prompts. Watch how the answer quality changes.

In [ ]:
def ask(system: str, user: str) -> str:
    r = client.messages.create(
        model=MODEL, max_tokens=256,
        system=system,
        messages=[{'role': 'user', 'content': user}]
    )
    return r.content[0].text

question = 'Should I use RAG or fine-tuning for my chatbot?'

poor_system = 'You are a helpful assistant.'

rich_system = (
    'You are an AI systems architect with 10 years of production ML experience. '
    'When asked about architecture decisions, always: '
    '(1) Ask clarifying questions about the use case, '
    '(2) Present trade-offs, not a single answer, '
    '(3) Give a concrete recommendation with reasoning.'
)

print('=== POOR CONTEXT ===')
print(ask(poor_system, question))
print()
print('=== RICH CONTEXT ===')
print(ask(rich_system, question))

## 2. Token Counting — See the Cost

Context engineering is context *economics*. Count tokens before and after adding context.

In [ ]:
import tiktoken

enc = tiktoken.get_encoding('cl100k_base')

def count(text: str) -> int:
    return len(enc.encode(text))

HAIKU_INPUT_PRICE = 0.80 / 1_000_000   # per token

contexts = {
    'Poor system prompt':   poor_system,
    'Rich system prompt':   rich_system,
    'User question':        question,
}

for name, text in contexts.items():
    tokens = count(text)
    cost   = tokens * HAIKU_INPUT_PRICE
    print(f'{name:30s}: {tokens:4d} tokens  (~${cost:.6f})')

## 3. The 11 CWA Layers — Quick Tour

Every token in a context window belongs to one of 11 layers.
Run this cell to see a minimal example of each.

In [ ]:
layers = [
    (1,  'Instructions (Identity)',   'You are a customer support agent for ACME Inc.'),
    (2,  'Safety & Guardrails',        'NEVER share customer data. NEVER make refunds directly.'),
    (3,  'Curated Knowledge',          'Shipping: 5-7 days. Returns: 30 days.'),
    (4,  'Task / Goal State',          'Current task: Process a return for order #12345.'),
    (5,  'Long-term Memory',           'Customer: VIP since 2020. Prefers email.'),
    (6,  'Short-term Memory',          '[Earlier] Customer asked about delayed order.'),
    (7,  'Tool Definitions',           'lookup_order(id), initiate_return(id, reason)'),
    (8,  'Dynamic RAG Results',        '[Retrieved] FedEx delays reported in Northeast.'),
    (9,  'Tool Results',               '[lookup_order] Status: delivered yesterday at 3pm.'),
    (10, 'Response Format',            'Respond with: 1) acknowledgement 2) resolution 3) CTA'),
    (11, 'User Query',                 'My package shows delivered but I never got it!'),
]

for num, name, example in layers:
    print(f'Layer {num:2d} — {name:30s}: {example[:60]}')

## 4. Try It Yourself

Assemble your own context window from the layers above and send it to Claude.

In [ ]:
system = '\n\n'.join([
    layers[0][2],   # Identity
    layers[1][2],   # Guardrails
    layers[2][2],   # Knowledge
    layers[4][2],   # Long-term memory
    layers[9][2],   # Response format
])

user_message = '\n\n'.join([
    layers[7][2],   # RAG results
    layers[8][2],   # Tool results
    layers[10][2],  # User query
])

answer = ask(system, user_message)
print(answer)